# Dynamic covariate balancing

This notebook is generated from the [Dynamic covariate balancing](https://moderndid.readthedocs.io/en/stable/user_guide/example_dyn_balancing.html) page of the ModernDiD documentation.

Democracy is not a treatment that a country adopts once and keeps for good. Over
six consecutive years, 12 of the 141 countries in the data move into or out of
democracy at least once. A country's status in a given year may also respond to
how its economy fared in the years before. The question here is whether a
stretch of democracy leaves a country with higher GDP per capita than the same
stretch under autocracy would.

Switching and feedback both break the difference-in-differences designs you'd
usually reach for. A staggered design assumes that each country adopts democracy
once and stays a democracy. Its parallel trends assumption also fails when
countries choose democracy in response to their past growth. A plain comparison
of democracies with autocracies fails for a different reason, because the
democracies in this data were far richer before the years being compared.
Instead, we'll use [`dyn_balancing`](https://moderndid.readthedocs.io/en/stable/api/generated/diddynamic/moderndid.diddynamic.dyn_balancing.html), the dynamic
covariate balancing estimator of [Viviano and Bradic
(2026)](https://doi.org/10.1093/biomet/asag016). It weights countries one year
at a time so that democracies and autocracies look alike in past GDP per capita
and, where the data allows, in region. It then compares their GDP per capita in
the final year.

The answer you'll reach is the effect of two years of democracy, against two
years of autocracy, on GDP per capita in the last year of the panel. On the way,
you'll see how closely the weights match the two groups of countries and how the
effect changes as the stretch grows from one year to five.

In [ ]:
!pip install -q "moderndid[all]==0.3.1"

In [ ]:
import polars as pl
from plotnine import options

options.figure_size = (8.7, 3.6)
options.dpi = 100
pl.Config.set_tbl_rows(20)

## Democracy and growth across 141 countries

The data comes from the panel that
[Acemoglu, Naidu, Restrepo, and Robinson (2019)](https://doi.org/10.1086/700936)
assembled to study whether democracy causes growth. Its outcome, log GDP per
capita, turns a gap between two groups of countries into a rough percent
difference in income. What suits the data to this method is how that study
treats democracy. It takes a country's democracy status in a given year to be as
good as random once its past GDP per capita and democracy are accounted for.
That assumption, known as sequential ignorability, lets democracy respond to how
the economy did in the past while ruling out hidden factors that drive both.

[`load_acemoglu`](https://moderndid.readthedocs.io/en/stable/api/generated/data/moderndid.load_acemoglu.html) loads the data with one row for each country in
each year. The first country's six years show how the columns fit together.

In [ ]:
import moderndid as did
import polars as pl

data = did.load_acemoglu()
data.select("Unit", "Time", "region", "D", "Y", "lag1.Value1", "lag2.Value1").head(6)

In each row, `Y` holds the country's log GDP per capita and `D` holds a 1 if the
country was a democracy that year. The row also records the year from 0 to 5 in
`Time` and one of seven world regions in `region`. The four columns from
`lag1.Value1` to `lag4.Value1` hold log GDP per capita one to four years
earlier.

In the last two years, 91 countries were democracies throughout and 46 were
autocracies throughout, against three that became democracies and one that
stopped being one. Before those two years began, the democracies' mean log GDP
per capita stood at 8.3 against 7.1 for the autocracies. That head start of 1.2
log points is what a plain comparison would mistake for an effect of democracy.

Four countries, two on each of those main histories, have no GDP per capita
figure for the last year. Since the estimator drops them with a warning, the
main estimates rest on 137 countries.

> **Regions where one history dominates**
>
> Since every industrialized country (INL) and every South Asian one (SAS) was a
> democracy in both of the last two years, no autocracy from those regions can
> stand in for them. The Middle East and North Africa (MNA) is the mirror image
> because 10 of its 11 countries were autocracies in both.

## The effect of a stretch of democracy

With so few switches, the comparison worth making is between whole stretches of
status at the end of the panel. For the last year of the panel, $T$, the target
is the gap between two average potential outcomes,

$$
\text{ATE} = \mathbb{E}\big[Y_T(D_{1:T-2},\, 1,\, 1)\big]
- \mathbb{E}\big[Y_T(D_{1:T-2},\, 0,\, 0)\big].
$$

In that display, $Y_T(D_{1:T-2}, d_{T-1}, d_T)$ is the log GDP per capita a
country would have in the last year with status $d_{T-1}$ and $d_T$ in the last
two years. Because every country keeps its own democracy history from before
that two-year window, the comparison averages over those earlier years.

Since each country followed only one history, its outcome under the other is
never observed. Filling that outcome in from countries on the other history
rests on four assumptions.

- A country's GDP per capita doesn't respond to its democracy status before that
  status takes effect.
- In each of the last two years, democracy is as good as random among countries
  that share a region, their GDP per capita in the four previous years, and any
  earlier status in the window.
- In each year of the window, a country's expected GDP per capita in the last
  year under either history is linear in its covariates and earlier status.
- Every country had some chance of being a democracy and some chance of being an
  autocracy in each year, given its covariates and earlier status.

[Dynamic covariate balancing DiD](https://moderndid.readthedocs.io/en/stable/background/diddynamic.html#background-diddynamic) states each
assumption formally and shows how the estimator recovers both potential outcomes
under them.

> **A different assumption from parallel trends**
>
> Nothing in these assumptions asks democracies and autocracies to follow parallel
> paths in GDP per capita. The comparison relies instead on region and past GDP per
> capita accounting for whatever ties a country's democracy to its growth.

## Pinning down the comparison

To estimate this comparison, we'll choose the treatment histories, the
covariates that account for treatment assignment, the outcome model and balance
requirements, and the method for measuring uncertainty.

In [ ]:
# Every choice sits in this dictionary so that each check can swap out one of them.
spec = dict(
    # The outcome, year, country, and democracy columns.
    yname="Y",
    tname="Time",
    idname="Unit",
    treatment_name="D",
    # Two years of democracy against two years of autocracy at the end of the panel.
    ds1=[1, 1],
    ds2=[0, 0],
    # Hold four years of past GDP per capita and the region fixed.
    xformla="~ lag1.Value1 + lag2.Value1 + lag3.Value1 + lag4.Value1",
    fixed_effects=["region"],
    # A lasso projection for GDP per capita with adaptive balance.
    method="lasso_plain",
    regularization=True,
    balancing="dcb",
    adaptive_balancing=True,
    # Analytic standard errors with Gaussian intervals.
    robust_quantile=False,
)

### Two years at the end of the panel

With `ds1=[1, 1]` and `ds2=[0, 0]`, the comparison sets two years of democracy
beside two years of autocracy. Two years is long enough for the dynamic part of
the method to matter. In the second year, the weights must match the weighted
averages from the first year rather than the plain averages of the whole sample.
Histories that switch status inside the window, such as autocracy followed by
democracy, hold too few countries to compare.

> **Count histories back from the final period**
>
> [`dyn_balancing`](https://moderndid.readthedocs.io/en/stable/api/generated/diddynamic/moderndid.diddynamic.dyn_balancing.html) lines up the last entry of `ds1` and
> `ds2` with `final_period`, the last year by default, and counts back one period
> per entry. That's why the time column must hold consecutive integers, as `Time`
> does here.

[Longer stretches of democracy](https://moderndid.readthedocs.io/en/stable/user_guide/example_dyn_balancing.html#longer-stretches-of-democracy) repeats the
comparison for stretches of one to five years.
[Stacking the earlier windows](https://moderndid.readthedocs.io/en/stable/user_guide/example_dyn_balancing.html#stacking-the-earlier-windows) instead adds every
earlier two-year window to the data.

### Four years of past GDP per capita and the region

Since democracy may respond to how the economy did, `xformla` holds fixed each
country's log GDP per capita in the four previous years. Four years follows the
preferred specification of the original study, as Viviano and Bradic (2026) do.
`fixed_effects=["region"]` adds a dummy for each of the seven regions on top of
them. Holding these fixed assumes that nothing else drives both democracy and
growth. [Leaving out past GDP per capita](https://moderndid.readthedocs.io/en/stable/user_guide/example_dyn_balancing.html#leaving-out-past-gdp-per-capita)
shows what happens when only the region stays.

> **Write dotted names as they are**
>
> `xformla` reads dotted names such as `lag1.Value1` as plain column names joined
> by `+`. Since a transformation such as `log(x)` raises an error, add the
> transformed values to the data as a column of their own.

Unlike Viviano and Bradic (2026), this example doesn't hold democracy in the
four previous years fixed. Nearly every country had the same status in those
years as in the window's first year. Holding it fixed would therefore push the
autocracy weights onto the handful of autocracies with a democratic past.
Leaving it out relies instead on past GDP per capita capturing whatever earlier
democracy did to the economy.

> **A covariate that democracy may have moved**
>
> In the second year, `lag1.Value1` holds a GDP per capita that democracy in the
> first year may already have moved. The second year's weights only compare
> countries that shared the first year's status. Balancing them on that outcome
> therefore adjusts for which countries kept their first-year status without
> removing the first year's effect.

### A lasso projection with adaptive balance

The [background page](https://moderndid.readthedocs.io/en/stable/background/diddynamic.html#background-diddynamic) sets out the estimator's two
stages in its sections on fitting the projections and on why the weights are
sequential. With `method="lasso_plain"` and `regularization=True`, the first
stage fits one linear model of GDP per capita to all countries at once and lets
cross-validation pick its lasso penalty. That shared model assumes that the
effects of democracy add up the same way in every country. Viviano and Bradic
(2026) use the linear model in their own analysis of the democracy panel. The
fully interacted model of `method="lasso_subsample"` drops that assumption at
the cost of fitting each year on fewer countries.

In the second stage, `balancing="dcb"` makes each year's weights as even as
possible while keeping each covariate's weighted mean close to its target.

> **Why inverse probability weights fail here**
>
> If you switch to `balancing="ipw"` or `"aipw"`, the call stops with an error on
> this data. Since the INL and SAS dummies predict democracy perfectly in the
> first year, the logistic model for the propensity score has no estimate. Without
> region, past GDP per capita still singles out the one country that left
> democracy in the second year.

With `adaptive_balancing=True`, the tolerance on each covariate's distance from
its target stays tight for covariates the lasso kept and widens for the rest
until weights can be found.
[Swapping the outcome model or the tolerance](https://moderndid.readthedocs.io/en/stable/user_guide/example_dyn_balancing.html#swapping-the-outcome-model-or-the-tolerance)
tries the fully interacted model, a nearly unpenalized ridge fit in place of the
lasso, and one tolerance for every covariate.

### Analytic standard errors with Gaussian intervals

The standard errors come from the analytic variance of the estimator. Because
that variance treats each country's first-year covariates as fixed, it measures
uncertainty about the effect for these particular countries. With
`robust_quantile=False`, the 95 percent interval uses the Gaussian critical
value that the inference theorem of Viviano and Bradic (2026) supports.
[Two other measures of uncertainty](https://moderndid.readthedocs.io/en/stable/user_guide/example_dyn_balancing.html#two-other-measures-of-uncertainty) tries
the larger chi-squared critical value and standard errors clustered by region.

The call below runs both stages for each history under `spec`.

In [ ]:
# Estimate mean GDP per capita in the last year under each history and the gap between them.
result = did.dyn_balancing(data, **spec)
print(result)

## Two years of democracy against two of autocracy

The first table reports an estimated change in log GDP per capita of −0.0244
under two years of democracy compared with two years of autocracy,
about 2.4 percent less. The 95 percent interval from −0.0639 to 0.0151 permits
changes ranging from about 6.2 percent less GDP per capita to 1.5 percent more
under the democracy history. Along with the p-value of 0.2259, that interval
gives us no reason to reject an effect of zero. Since it still allows changes in
either direction, it doesn't establish that democracy has no effect.

The effect is the gap between the two means below the table. Had all 137
countries been democracies in both years, `mu(ds1)` puts their mean log GDP per
capita in the last year at 7.8179, against 7.8423 under `mu(ds2)` for two years
of autocracy.

### How closely the weights match the two groups

Each set of weights tries to make the countries on its history look like all 137
countries in every covariate. The `imbalances` field of the result measures how
far each one falls short, as the gap between a covariate's weighted mean and its
target in standard deviations of the covariate. The autocracy weights in the
first year show where those matches fall short.

In [ ]:
# Read the first year's diagnostic directly from the result.
result.imbalances["ds2"].filter(pl.col("period") == 1).select("covariate", "imbalance")

The democracy weights match every covariate to within 0.0013 standard deviations
in both years. The autocracy weights come close on past GDP per capita too,
where the largest gap is −0.0150 for `lag4.Value1`. The region dummies are where
the autocracy weights fall short of their targets. The two regions without
autocracies, INL and SAS, sit at −0.4707 and −0.1939. MNA at 0.6870 and EAP,
East Asia and the Pacific, at 0.5105 end up overrepresented instead.

Those gaps are allowed to stand because the lasso kept only `lag1.Value1`.
Adaptive balancing held that one covariate to the tight tolerance and let the
tolerance on every other covariate, the three older lags included, widen until
weights could be found. The older lags still come out well balanced because
they move almost in step with `lag1.Value1`.

> **Overlap fails for INL and SAS countries**
>
> The autocracy weights give INL and SAS countries no weight at all. Their GDP per
> capita under autocracy comes from the outcome model alone. That estimate holds
> only if region really adds nothing once the previous year's GDP per capita is
> known, as the lasso found.

## Longer stretches of democracy

If democracy raises GDP per capita only slowly, two years may be too short to
show it. Passing `histories_length` makes
[`dyn_balancing`](https://moderndid.readthedocs.io/en/stable/api/generated/diddynamic/moderndid.diddynamic.dyn_balancing.html) repeat the comparison for each
length in the list. A run of length $h$ keeps only the last $h$ entries of `ds1`
and `ds2` and estimates the target above with the last $h$ years in place of
the last two.

In [ ]:
# One to five years of democracy against as many of autocracy, all ending in the last year.
history = did.dyn_balancing(
    data, **(spec | {"ds1": [1] * 5, "ds2": [0] * 5, "histories_length": [1, 2, 3, 4, 5]})
)
print(history)

One year of democracy changes log GDP per capita by −0.0021. Longer stretches
pull the estimate down to −0.0312 at five years. The standard error doubles from
0.0149 at one year to 0.0297 at five. Each extra year adds another round of
weights and another term to the variance.

In the figure that [`plot_dyn_balancing_history`](https://moderndid.readthedocs.io/en/stable/api/generated/plotting/moderndid.plots.plot_dyn_balancing_history.html) draws
from these estimates, the bars widen with each added year.

In [ ]:
# Bars mark each length's 95 percent interval and the dashed line an effect of zero.
did.plot_dyn_balancing_history(history) + did.theme_moderndid()

Since every bar in the figure crosses zero, none of the five stretches has an
effect that the data can tell apart from zero.

> **One comparison per history length**
>
> Read the points as five separate comparisons for the same 137 countries rather
> than one effect unfolding over time. The countries that supply each observed
> history change as the window lengthens because fewer kept one status for that
> long.

## Which choices move the answer

The interval around the two-year effect leaves room for changes in either
direction.

### Stacking the earlier windows

With `pooled=True`, every earlier two-year window of the panel joins the
comparison as a separate country history. Because the windows end in different
years, the check also adds `Time` to `fixed_effects` to give each year its own
dummy, as the pooled regression of Viviano and Bradic (2026) does. A shock that
hits every country in the same year can then be absorbed by that year's dummy.

In [ ]:
# Stack every earlier two-year window as its own country history and give each year a dummy.
pooled = did.dyn_balancing(data, **(spec | {"pooled": True, "fixed_effects": ["region", "Time"]}))

In [ ]:
print(pooled)

The report now counts 141 countries and 701 stacked histories, since the four
countries missing the last year's figure still contribute their earlier windows.
The effect moves to −0.0158 and its interval narrows to run from −0.0404 to
0.0089. Because the same country appears in several windows, the standard errors
now cluster by country. Pooling buys that precision by assuming that the
two-year effect and the outcome model are the same in every window apart from
each year's own intercept.

### Leaving out past GDP per capita

To see how much of the answer past GDP per capita carries, this check drops the
four lags from `xformla` and keeps only the region dummies.

In [ ]:
# The same specification without past GDP per capita, so only the region is balanced.
without_past_gdp = did.dyn_balancing(data, **(spec | {"xformla": None}))
print(f"effect {without_past_gdp.att:.4f}, standard error {without_past_gdp.se:.4f}")

The effect jumps from −0.0244 to 0.3067, or about 36 percent more GDP per capita
for the democracy history. Its standard error rises from 0.0201 to 0.2153
because the outcome model no longer knows any past GDP per capita.

> **Balancing only region keeps the head start**
>
> The democracies were already 1.2 log points richer before the last two years
> began. A comparison that balances only region carries the part of that head
> start that region doesn't explain into the estimate of 0.3067.

### Swapping the outcome model or the tolerance

This check turns to the settings that shape each of the estimator's two stages,
one at a time.

In [ ]:
# The two-year effect when only the outcome model or the balance tolerance changes.
interacted = did.dyn_balancing(data, **(spec | {"method": "lasso_subsample"}))
ridge = did.dyn_balancing(data, **(spec | {"regularization": False}))
one_tolerance = did.dyn_balancing(data, **(spec | {"adaptive_balancing": False}))
print(f"fully interacted: effect {interacted.att:.4f}, standard error {interacted.se:.4f}")
print(f"ridge: effect {ridge.att:.4f}, standard error {ridge.se:.4f}")
print(f"one tolerance: effect {one_tolerance.att:.4f}, standard error {one_tolerance.se:.4f}")

Under these swaps the effect runs from −0.0238 to −0.0121, a spread smaller than
the standard error of 0.0201 in our specification. The fully interacted model
and ridge report smaller standard errors of 0.0129 and 0.0105. Because both fit
GDP per capita more closely than a lasso that keeps only `lag1.Value1`, their
residuals in the last year and the year-to-year changes in their projections
both come out smaller.

The first-year gaps that the autocracy weights leave under a single tolerance
show what adaptive balancing protects.

In [ ]:
# The first year's gaps when a single tolerance covers every covariate.
one_tolerance.imbalances["ds2"].filter(pl.col("period") == 1).select("covariate", "imbalance")

When one tolerance applies to every covariate, the INL gap forces it wide enough
to leave past GDP per capita off by as much as 0.1220 standard deviations,
against 0.0150 in our specification. Ridge leaves the same gaps, since none of
its coefficients is zero and adaptive balancing therefore has no covariate to
loosen.

### Two other measures of uncertainty

Only the measure of uncertainty differs under the two settings below. With
`robust_quantile=True`, the interval uses the square root of a chi-squared
critical value with twice as many degrees of freedom as the history has years.
With `clustervars=["region"]`, the standard error lets countries in the same
region share shocks.

In [ ]:
# The same estimate under a chi-squared critical value and under region clusters.
chi_squared = did.dyn_balancing(data, **(spec | {"robust_quantile": True}))
region_clusters = did.dyn_balancing(data, **(spec | {"clustervars": ["region"]}))
print(f"chi-squared critical value {chi_squared.robust_quantile:.4f}")
print(f"standard error with region clusters {region_clusters.se:.4f}")

# The ATE row of each fit, with the interval its inference settings give.
ate = pl.col("parameter") == "ATE"
pl.concat([did.to_df(chi_squared).filter(ate), did.to_df(region_clusters).filter(ate)])

The chi-squared critical value of 3.0802 widens the interval to run from −0.0864
to 0.0377. Clustering by region raises the standard error from 0.0201 to 0.0348
and stretches the interval to span −0.0925 to 0.0438.

> **Seven clusters make a fragile standard error**
>
> With seven regions, the clustered variance rests on seven cluster sums and has
> no small-sample correction. Treat the larger standard error as a hint that
> regional shocks could matter rather than as a measure of how much.

Every check that keeps past GDP per capita among the covariates leaves the
two-year effect between −0.0244 and −0.0121 with an interval that covers zero.
Only leaving out past GDP per capita moves it far, to 0.3067. Sequential
ignorability given region and past GDP per capita still carries every estimate
on this page. A hidden factor that moved democracy and growth together would
bias all of them. If you'd rather assume parallel trends among countries that
start from the same status, the
[intertemporal treatment example](https://moderndid.readthedocs.io/en/stable/user_guide/example_inter_did.html#example-inter-did) estimates the effects
of a treatment that switches on and off under that assumption.